# NoisiQ — Week 10 Notebook
**Noise-Aware Quantum Circuit Simulation and Visualization**
*Period: May 25 – May 31, 2026*

---

## Purpose of This Notebook

This notebook serves as the living record and demonstration for Week 10 — the final week
of the NoisiQ project. It documents the polish pass, final validation, and PyPI publication.

By the end of this notebook you should be able to:
- Install `noisiq` directly from PyPI (`pip install noisiq`)
- Walk through the complete end-to-end workflow: import → noise → animate → heatmap → suppress → compare
- Confirm the hover system cleanly switches between single-shot and many-shot fields on all circuit sizes
- Confirm the fan-out graph opens correctly from the hover menu
- See the full package with docs, badges, and a working public API

---

## Week 10 Milestone Summary

Week 10 is the **polish and ship** week. The final deliverable is a publicly installable
package with a comprehensive demo notebook and a presentation-ready report.

---

## Status Tracker

| Task | Owner | Status |
|------|-------|--------|
| `pyproject.toml` — finalize metadata, classifiers, version `0.1.0` stable | DS | ☐ Todo |
| Full docstring pass — all public classes and functions | TJ | ☐ Todo |
| `docs/` — Getting Started + API reference (pdoc or mkdocs-material) | TJ | ☐ Todo |
| Performance profiling — many-shot runner benchmark at 10/20/50 qubits | TJ | ☐ Todo |
| Seamless display QA — hover single/many-shot dispatch on all circuit sizes | TJ | ☐ Todo |
| Fan-out hover integration QA — opens cleanly from hover menu | TJ | ☐ Todo |
| Full test suite passing — all tests green, CI passing | TJ | ☐ Todo |
| GIF + HTML export verified end-to-end | TJ | ☐ Todo |
| `weekly_notebooks/final_demo_notebook.ipynb` — comprehensive end-to-end demo | DS | ☐ Todo |
| TestPyPI publish test | DS | ☐ Todo |
| PyPI publish — `twine upload dist/*` | DS | ☐ Todo |
| `README.md` — final update with badges + screenshots | DS | ☐ Todo |

---

## File Build Order

```
1. All modules — docstring pass (no new files, just edits)
2. pyproject.toml — metadata finalization + version bump
3. docs/ — auto-generated from docstrings via pdoc/mkdocs
4. QA pass: hover system (single/many-shot dispatch) + fan-out graph trigger
5. Performance profiling script — benchmark + fix bottlenecks
6. final_demo_notebook.ipynb — comprehensive end-to-end demo
7. python -m build → dist/
8. twine check dist/*
9. twine upload --repository testpypi dist/*
10. twine upload dist/*
11. README.md — final update
```

---

## Technical Requirements for Week 10

**Seamless display QA checklist:**
- Single-shot hover shows: gate type, exact error, incoming/outgoing states
- Many-shot hover shows: gate type, error probability distribution, most likely error, fidelity estimate
- Switching from single-shot to many-shot run updates hover fields automatically (no restart needed)
- Fan-out graph button present in both modes; opens new matplotlib figure on click
- Tested on: 1-qubit trivial circuit, 5-qubit GHZ, 10-qubit random Clifford

**PyPI metadata checklist (`pyproject.toml`):**
- `name = "noisiq"`, `version = "0.1.0"`
- `description` — one-sentence summary
- `readme = "README.md"`, `license = {text = "MIT"}`
- `classifiers` — `Development Status :: 3 - Alpha`, `Topic :: Scientific/Engineering :: Physics`
- `keywords` — `quantum`, `noise`, `simulation`, `visualization`
- `urls` — homepage, repository, documentation

**Performance benchmark targets:**
- 10-qubit circuit, 1000 shots: < 1 second
- 20-qubit Clifford circuit, 1000 shots (Stim backend): < 5 seconds

---

## Notes and Decisions Log

| Date | Note | Name |
|------|------|------|
| 2026-05-25 | Week 10 started. Polish and ship. | DS |
| 2026-04-21 | Added seamless display QA and fan-out hover integration QA as explicit checklist items. | DS |
| | | |

# Installation — After PyPI Publication

```bash
pip install noisiq
```

With optional Aer backend:
```bash
pip install noisiq[aer]
```

In [ ]:
# Final end-to-end demo: full NoisiQ workflow
import noisiq as nq
from qiskit.circuit.library import QFT

print(f"noisiq {nq.__version__} — installed from PyPI")

# === Step 1: Import a Qiskit circuit ===
qc = QFT(4).decompose()
circuit = nq.adapters.from_qiskit(qc)
print(f"Imported: {circuit.n_qubits}-qubit QFT, {len(circuit.operations)} gates")

# === Step 2: Select a noise model ===
preset = nq.noise.SuperconductingPreset()
noise = preset.to_noise_model()
print(f"Noise model: {noise.describe()}")

# === Step 3: Run many-shot simulation ===
backend = nq.backends.BackendSelector.select(circuit, noise)
runner = nq.backends.ManyShotRunner(circuit, noise_model=noise, n_shots=1000)
result_before = runner.run()

# === Step 4: Visualize error heatmap ===
nq.visualization.plot_error_heatmap(result_before, circuit, title="QFT — Superconducting Noise")

# === Step 5: Apply noise suppression ===
pipeline = nq.suppression.SuppressionPipeline([
    nq.suppression.HahnEcho(qubit=0, idle_time=1),
    nq.suppression.HahnEcho(qubit=1, idle_time=1),
])
suppressed_circuit, _ = pipeline.apply(circuit)
result_after = nq.backends.ManyShotRunner(suppressed_circuit, noise_model=noise, n_shots=1000).run()

# === Step 6: Before/after comparison ===
nq.visualization.plot_comparison(
    before=result_before,
    after=result_after,
    circuit=circuit,
    title="Hahn Echo suppression on QFT circuit"
)

# === Step 7: Export animation ===
single_shot = nq.backends.StimBackend().run(circuit, noise_model=noise)
anim = nq.visualization.animate_propagation(single_shot, circuit)
nq.visualization.export_gif(anim, path="noisiq_demo.gif")
print("Exported noisiq_demo.gif")

In [ ]:
# Performance benchmark
import time

for n_qubits in [10, 20]:
    bench_circuit = nq.Circuit(n_qubits=n_qubits, name=f"bench_{n_qubits}q")
    for i in range(n_qubits):
        bench_circuit.add_gate(nq.ir.H, qubits=[i])
    for i in range(n_qubits - 1):
        bench_circuit.add_gate(nq.ir.CNOT, qubits=[i, i + 1])

    noise = nq.noise.DepolarizingChannel(p=0.01)
    runner = nq.backends.ManyShotRunner(bench_circuit, noise_model=noise, n_shots=1000)

    start = time.perf_counter()
    runner.run()
    elapsed = time.perf_counter() - start
    print(f"{n_qubits}-qubit, 1000 shots: {elapsed:.3f}s")